## EEG Segmentation from .xdf file
This notebook is responsible for dividing one EEG session into its segments, meaning that each segment corresponds to one genre listens to. The input for this is an .xdf file from an EEG session. The output will be a directory with new .xdf files each representing one song of the whole session

In [40]:
# Setup path to data directory
PATH_TO_DATA = "../../data"
SONG_COUNT = 50
SESSION_NUMBER = "S005"

In [56]:
# Imports
import pyxdf
import numpy as np
import pandas as pd
import os
from scipy.io import savemat

In [42]:
def setup(filename):
  data, header = pyxdf.load_xdf(f"{PATH_TO_DATA}/{filename}")
  return data, header

In [43]:
data, header = setup("sub-P001_ses-S005_task-Default_run-001_eeg.xdf")

In [44]:
def find_stream(stream_name):
  for stream in data:
    if stream["info"]["name"] == [stream_name]:
      print(f"Found the stream {stream_name}!")
      return stream
  print("Couldn't find the stream.")

In [45]:
# Get all the streams
eeg_stream = find_stream("Headset_EEG_DR250")
music_markers_stream = find_stream("MusicMarkers")
song_stream = find_stream("Song")

Found the stream Headset_EEG_DR250!
Found the stream MusicMarkers!
Found the stream Song!


In [46]:
# Get timestamps and data
# Annotation: data holds the real values; timestamps the corresponding time the specific value was measured at
eeg_data = eeg_stream["time_series"]
eeg_timestamps = eeg_stream["time_stamps"]

music_markers_data = music_markers_stream["time_series"]
music_markers_timestamps = music_markers_stream["time_stamps"]

song_data = song_stream["time_series"]
song_timestamps = song_stream["time_stamps"]

In [47]:
# Remove the 10 other channels
eeg_data_processed = eeg_data[:, :8]

In [48]:
np.shape(eeg_timestamps)

(453732,)

In [49]:
# Read out EEG channel data
channel_data_acc = [] # stores data over the course of all channels and each song i guess idk
time_stamps = [] # stores the timestamps; should have the same size as channel_data_acc
pointer = 0 # pointer that iterates over all the timestamps/data
for i in range(0, SONG_COUNT*2, 2):
  # get the start and end timestamp of the song
  print(f"======START {i}=====")
  start = music_markers_timestamps[i]
  print(f"Start timestamp: {start}")
  end = music_markers_timestamps[i+1]
  print(f"Start timestamp: {end}")

  # find the index in which the timestamp can be found
  while True:
    if eeg_timestamps[pointer] >= start: # found the start timestamp or rather got the closest one
      break
    pointer += 1
  print(f"The pointer is {pointer}")
  pointer_buf = pointer # store the pointer since it is increased steadily

  print("START READING OUT SIGNAL DATA")
  song_eeg_data = [] # the eeg data from this specific song (song i)
  for j in range(8):
    print(f"Channel {j} is being read out")
    channel_data = [] # signal data from the channel that is being read out rn
    current_channel = eeg_data_processed[:, j] # get the current channel
    while eeg_timestamps[pointer] < end:
      # append data to the channel data list
      channel_data.append(current_channel[pointer])
      pointer += 1
    pointer = pointer_buf # reset the pointer
    song_eeg_data.append(channel_data) # append the channel data to the current song's signal data
  channel_data_acc.append(song_eeg_data)
  print(f"======END {i}=====")

======START 0=====
Start timestamp: 12633.674961007178
Start timestamp: 12663.938025377729
The pointer is 9754
START READING OUT SIGNAL DATA
Channel 0 is being read out
Channel 1 is being read out
Channel 2 is being read out
Channel 3 is being read out
Channel 4 is being read out
Channel 5 is being read out
Channel 6 is being read out
Channel 7 is being read out
======END 0=====
======START 2=====
Start timestamp: 12668.94146177286
Start timestamp: 12699.066812843543
The pointer is 18556
START READING OUT SIGNAL DATA
Channel 0 is being read out
Channel 1 is being read out
Channel 2 is being read out
Channel 3 is being read out
Channel 4 is being read out
Channel 5 is being read out
Channel 6 is being read out
Channel 7 is being read out
======END 2=====
======START 4=====
Start timestamp: 12704.070400538676
Start timestamp: 12734.558798109007
The pointer is 27323
START READING OUT SIGNAL DATA
Channel 0 is being read out
Channel 1 is being read out
Channel 2 is being read out
Channel 3 

In [50]:
# Print all shapes of the eeg data
for i in range(SONG_COUNT):
  print(f"This is the shape of song {i+1}: {np.shape(channel_data_acc[i])}")

This is the shape of song 1: (8, 7553)
This is the shape of song 2: (8, 7518)
This is the shape of song 3: (8, 7609)
This is the shape of song 4: (8, 7519)
This is the shape of song 5: (8, 7521)
This is the shape of song 6: (8, 7519)
This is the shape of song 7: (8, 7521)
This is the shape of song 8: (8, 7519)
This is the shape of song 9: (8, 7520)
This is the shape of song 10: (8, 7518)
This is the shape of song 11: (8, 7519)
This is the shape of song 12: (8, 7519)
This is the shape of song 13: (8, 7522)
This is the shape of song 14: (8, 7519)
This is the shape of song 15: (8, 7519)
This is the shape of song 16: (8, 7519)
This is the shape of song 17: (8, 7521)
This is the shape of song 18: (8, 7522)
This is the shape of song 19: (8, 7517)
This is the shape of song 20: (8, 7519)
This is the shape of song 21: (8, 7519)
This is the shape of song 22: (8, 7553)
This is the shape of song 23: (8, 7520)
This is the shape of song 24: (8, 7524)
This is the shape of song 25: (8, 7519)
This is t

In [51]:
table = {}
# now here comes the tricky part: we have to store everything in a dictionary
# how do we want our data structured?
# as rows, we want the song index
rows = []
for i in range(SONG_COUNT):
  rows.append(f"Song{i}")

# as columns, we obviously want the data. the column names should suggest the name of the Channel. to keep it simple, we can just have <BIOSIGNAL>Ch<CHANNEL NO.>
eeg_channels = [f"EEG_Ch{i+1}" for i in range(8)]

# the genres are also columns which we can already initialize
table["genre"] = []
for i in range(len(song_data)-1):
  table["genre"].append(song_data[i][1])

# initialize the rest of the columns in the table
for channel in eeg_channels:
  table[channel] = []

# now we have to construct the actual values of the columns
for song_counter in range(SONG_COUNT):
  current_song = channel_data_acc[song_counter]
  counter = 0
  for channel_data in current_song:
    column_name = eeg_channels[counter]
    counter += 1
    table[column_name].append(channel_data)

df = pd.DataFrame(table)
df.index = rows
df
  


,genre,EEG_Ch1,EEG_Ch2,EEG_Ch3,EEG_Ch4,EEG_Ch5,EEG_Ch6,EEG_Ch7,EEG_Ch8
Song0,disco,"[-42939.29, -42943.043, -42945.258, -42942.64,...","[105617.695, 105613.2, 105610.73, 105613.47, 1...","[-69.1956, -69.3297, -69.4638, -69.39675, -69....","[47222.914, 47220.098, 47216.14, 47213.727, 47...","[-47914.734, -47917.016, -47919.832, -47919.69...","[194512.25, 194476.38, 194449.36, 194481.94, 1...","[38004.14, 38007.562, 38006.89, 38010.242, 380...","[230558.67, 230534.6, 230527.16, 230549.69, 23..."
Song1,rock,"[-41460.3, -41463.316, -41459.293, -41456.277,...","[108742.09, 108740.28, 108742.56, 108745.11, 1...","[-69.12855, -69.0615, -68.99445, -68.99445, -6...","[45449.44, 45446.758, 45449.574, 45452.59, 454...","[-47050.06, -47052.473, -47051.668, -47047.58,...","[192991.28, 192982.7, 193002.62, 193059.81, 19...","[38412.008, 38411.336, 38413.48, 38416.566, 38...","[229547.42, 229535.22, 229557.47, 229599.25, 2..."
Song2,hiphop,"[-39740.133, -39736.914, -39738.12, -39739.664...","[111998.24, 111993.68, 111986.91, 111984.695, ...","[-69.12855, -68.7933, -68.86035, -69.0615, -68...","[44416.668, 44418.883, 44413.52, 44409.76, 444...","[-45541.367, -45542.305, -45546.996, -45543.98...","[191630.11, 191612.34, 191581.5, 191583.44, 19...","[38909.383, 38913.676, 38909.383, 38907.37, 38...","[229034.08, 229022.62, 229008.34, 229008.47, 2..."
Song3,reggae,"[-38513.117, -38513.25, -38513.453, -38511.844...","[114649.664, 114649.734, 114643.03, 114645.914...","[-69.12855, -69.3297, -69.1956, -68.99445, -69...","[43363.918, 43362.375, 43356.34, 43351.78, 433...","[-44609.57, -44610.914, -44612.926, -44610.31,...","[189824.72, 189855.03, 189840.75, 189811.38, 1...","[38851.184, 38850.848, 38847.766, 38849.17, 38...","[227702.88, 227742.9, 227729.02, 227701.86, 22..."
Song4,pop,"[-36970.832, -36965.203, -36973.383, -36976.66...","[117533.555, 117544.484, 117534.56, 117527.19,...","[-68.9274, -69.1956, -69.3297, -69.26265, -69....","[43281.246, 43292.91, 43285.938, 43283.727, 43...","[-43553.266, -43548.305, -43552.594, -43556.48...","[189038.02, 189126.27, 189134.9, 189071.48, 18...","[39131.25, 39136.95, 39132.996, 39130.043, 391...","[227417.84, 227509.3, 227519.36, 227444.39, 22..."
Song5,reggae,"[-35698.895, -35697.824, -35697.152, -35697.55...","[120036.87, 120040.95, 120048.195, 120050.21, ...","[-69.1956, -69.53085, -69.4638, -69.4638, -69....","[42624.89, 42630.656, 42632.87, 42635.285, 426...","[-42708.1, -42711.117, -42715.207, -42716.684,...","[187178.45, 187152.17, 187192.14, 187225.86, 1...","[39219.625, 39217.613, 39216.742, 39216.27, 39...","[226333.17, 226307.03, 226351.95, 226391.23, 2..."
Song6,jazz,"[-34456.793, -34456.324, -34458.805, -34458.60...","[122527.04, 122524.29, 122516.445, 122519.39, ...","[-69.26265, -69.39675, -69.12855, -69.1956, -6...","[41610.426, 41606.47, 41597.887, 41598.02, 416...","[-41841.21, -41841.01, -41843.223, -41840.676,...","[185910.4, 185909.88, 185867.16, 185866.48, 18...","[39190.12, 39188.51, 39185.227, 39185.094, 391...","[225774.31, 225779.2, 225735.16, 225721.08, 22..."
Song7,disco,"[-33586.15, -33583.734, -33581.39, -33579.848,...","[124323.98, 124325.65, 124332.42, 124335.914, ...","[-69.3297, -69.53085, -69.3297, -69.3297, -69....","[39969.71, 39966.562, 39970.45, 39971.86, 3996...","[-41244.062, -41242.723, -41242.926, -41244.13...","[184259.97, 184233.55, 184244.61, 184335.06, 1...","[39072.984, 39073.656, 39072.984, 39071.11, 39...","[223901.81, 223887.53, 223892.36, 223954.52, 2..."
Song8,country,"[-32552.775, -32553.512, -32553.445, -32556.19...","[126360.89, 126355.93, 126352.51, 126347.27, 1...","[-69.3297, -69.1956, -69.53085, -69.732, -69.2...","[40071.492, 40065.324, 40066.8, 40065.39, 4006...","[-40411.504, -40412.375, -40409.695, -40410.97...","[183112.75, 183117.9, 183082.64, 183034.83, 18...","[39272.93, 39269.777, 39271.79, 39268.77, 3926...","[222564.62, 222551.95, 222519.44, 222472.77, 2..."
Song9,jazz,"[-31038.049, -31038.584, -31038.584, -31041.06...","[128881.766,

In [60]:
# Save the table
os.makedirs(f"{PATH_TO_DATA}/signal_data", exist_ok=True)

# create a folder for numpy data
os.makedirs(f"{PATH_TO_DATA}/signal_data/numpy", exist_ok=True)
np.save(f"{PATH_TO_DATA}/signal_data/numpy/{SESSION_NUMBER}.npy", df.to_numpy())
# create a folder for matlab data
os.makedirs(f"{PATH_TO_DATA}/signal_data/matlab", exist_ok=True)
savemat(f"{PATH_TO_DATA}/signal_data/matlab/{SESSION_NUMBER}.mat", table)


## Final result

# DOODLE: REMOVE ME

In [52]:
# Example for multindex
bre = {
    'Sales': [15000, 12000, 8000, 9500, 20000, 18000, 11000, 13500, 
              7500, 9000, 14000, 16000],
    'Units_Sold': [150, 120, 80, 95, 200, 180, 110, 135, 
                   75, 90, 140, 160],
    'Profit_Margin': [0.25, 0.30, 0.35, 0.28, 0.22, 0.27, 0.32, 0.29,
                      0.40, 0.35, 0.26, 0.24]
}
regions = ['North', 'North', 'North', 'North', 'South', 'South', 
           'South', 'South', 'West', 'West', 'West', 'West']
cities = ['New York', 'New York', 'Boston', 'Boston', 'Miami', 'Miami',
          'Atlanta', 'Atlanta', 'LA', 'LA', 'Seattle', 'Seattle']
products = ['Laptops', 'Phones', 'Laptops', 'Phones', 'Laptops', 'Phones',
            'Laptops', 'Phones', 'Laptops', 'Phones', 'Laptops', 'Phones']

# Create the MultiIndex
multi_index = pd.MultiIndex.from_arrays([regions, cities, products], 
                                        names=['Region', 'City', 'Product'])

# Create DataFrame with MultiIndex
df = pd.DataFrame(bre, index=multi_index)

In [53]:
df

Sales  Units_Sold  Profit_Margin
Region City     Product                                  
North  New York Laptops  15000         150           0.25
                Phones   12000         120           0.30
       Boston   Laptops   8000          80           0.35
                Phones    9500          95           0.28
South  Miami    Laptops  20000         200           0.22
                Phones   18000         180           0.27
       Atlanta  Laptops  11000         110           0.32
                Phones   13500         135           0.29
West   LA       Laptops   7500          75           0.40
                Phones    9000          90           0.35
       Seattle  Laptops  14000         140           0.26
                Phones   16000         160           0.24